# Phase 3 — Empirical Analysis

**Research question:** Can stablecoins replace the legacy correspondent-banking (SWIFT) network for cross-border value transfer?

This notebook tests four hypotheses using the master datasets built in Phase 2B.
Each hypothesis section is self-contained: data load, specification, estimation, diagnostics, and output.

**Execution order:** H1 (Metcalfe) → H3 (HHI concentration) → H2 (diffusion panel) → H4 (cost friction).

See `Master_Recovery_Roadmap.md` §3.2–3.5 for methodology and `docs/DATA_DICTIONARY.md` for column definitions.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Econometrics
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import het_breuschpagan
from linearmodels.panel import PanelOLS, PooledOLS
from scipy import stats

# ══════════════════════════════════════════════════════════════════════
# GLOBAL CONFIGURATION
# ══════════════════════════════════════════════════════════════════════
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Paths
DATA_DIR = '../data/03_processed/'
FIG_DIR  = '../outputs/figures/'
TBL_DIR  = '../outputs/tables/'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TBL_DIR, exist_ok=True)

# Plot defaults
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style('whitegrid')

print(f'Random seed: {RANDOM_SEED}')
print(f'Outputs: figures → {FIG_DIR}, tables → {TBL_DIR}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

def run_adf(series, name, significance=0.05):
    """Run ADF test and return a summary dict."""
    result = adfuller(series.dropna(), autolag='AIC')
    return {
        'series': name,
        'adf_stat': result[0],
        'p_value': result[1],
        'lags_used': result[2],
        'n_obs': result[3],
        'critical_5pct': result[4]['5%'],
        'stationary': result[1] < significance,
    }


def wald_test(model, hypothesis, name):
    """Run a Wald test on a fitted statsmodels OLS result."""
    t = model.t_test(hypothesis)
    return {
        'test': name,
        'statistic': float(t.statistic),
        'p_value': float(t.pvalue),
    }


def save_fig(fig, filename):
    """Save figure to outputs/figures/ at 300 DPI."""
    path = os.path.join(FIG_DIR, filename)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved: {path}')


def save_table(df, basename):
    """Save table as both CSV and LaTeX."""
    csv_path = os.path.join(TBL_DIR, f'{basename}.csv')
    tex_path = os.path.join(TBL_DIR, f'{basename}.tex')
    df.to_csv(csv_path)
    df.to_latex(tex_path, float_format='%.4f')
    print(f'Saved: {csv_path}, {tex_path}')


print('Helper functions loaded.')

---
# H1 — Network Effects (Metcalfe's Law)

**Hypothesis:** Stablecoin transfer activity (count) scales super-linearly with active addresses (β > 1).

**Specification:** `log(transfer_count) = α + β × log(active_addresses) + ε`

**Tests:** Wald tests for β = 1 (linear scaling) and β = 2 (strict Metcalfe). Per-asset, not pooled.

**Pre-checks:** ADF stationarity → Engle-Granger cointegration if needed → Newey-West SE.

**Note on DV:** Transfer count (not value) is used because CoinMetrics Community tier does not
expose `TxTfrValAdjUSD`. Count-based β typically lands 1.3–1.7 in the literature (Peterson 2018,
Wheatley et al. 2018), below the strict value-based prediction of 2.

In [ ]:
# ── H1: Load data ─────────────────────────────────────────────────────
h1 = pd.read_csv(os.path.join(DATA_DIR, 'h1_network_effects.csv'), parse_dates=['date'])
print(f'H1 loaded: {h1.shape[0]} rows, assets: {h1["asset"].unique()}')
h1.head()

In [ ]:
# ── H1: Stationarity tests (ADF) ─────────────────────────────────────
# TODO: Phase 3.2 — ADF on log(active_addresses) and log(transfer_count)
# for USDC and USDT separately. Summarize results and escalate to
# Claude.ai if specification choice (levels vs first-differences) is non-obvious.

In [ ]:
# ── H1: Cointegration test (if needed) ────────────────────────────────
# TODO: Phase 3.2 — Engle-Granger cointegration if series are non-stationary.

In [ ]:
# ── H1: OLS estimation ────────────────────────────────────────────────
# TODO: Phase 3.2 — Log-log OLS with Newey-West SE, per asset.
# Wald tests: β=1, β=2. Report β, SE, 95% CI, p-values, R².

In [ ]:
# ── H1: Metcalfe scatter plots ────────────────────────────────────────
# TODO: Phase 3.2 — Log-log scatter with fitted line per asset.
# Save: h1_metcalfe_usdc.png, h1_metcalfe_usdt.png

---
# H3 — Market Concentration (HHI)

**Hypothesis:** The stablecoin market exhibits winner-takes-all dynamics (rising HHI).

**Test:** OLS trend of HHI over time with Newey-West SE.

**Key events to annotate:** Terra/UST collapse (May 2022), FTX collapse (Nov 2022), SVB collapse (Mar 2023).

**Note:** Read `data/02_intermediate/h3_diagnostic_report.md` Phase 3/4 narrative notes before
writing interpretation. The USDT share U-shape and conditional concentration dynamics framing
are documented there.

In [ ]:
# ── H3: Load data ─────────────────────────────────────────────────────
h3 = pd.read_csv(os.path.join(DATA_DIR, 'h3_concentration.csv'), parse_dates=['date'])
print(f'H3 loaded: {h3.shape[0]} rows, date range: {h3["date"].min()} to {h3["date"].max()}')
h3.head()

In [ ]:
# ── H3: HHI time series chart ─────────────────────────────────────────
# TODO: Phase 3.3 — Plot hhi_full and hhi_top5, annotate Terra/FTX/SVB events.
# Save: h3_hhi_timeseries.png

In [ ]:
# ── H3: Trend test (OLS + Newey-West) ─────────────────────────────────
# TODO: Phase 3.3 — OLS of HHI on time trend. Report coefficient, SE, p-value.
# Top-3 stablecoins at start and end of window.

In [ ]:
# ── H3: Summary table ─────────────────────────────────────────────────
# TODO: Phase 3.3 — Save h3_top_stablecoins.csv

---
# H2 — Country-Level Diffusion & Institutional Gaps

**Hypothesis:** Crypto adoption accelerates in countries with weak banking infrastructure,
and the relationship strengthens post-Nov 2022.

**Specification sequence:**
1. Pooled OLS (baseline)
2. Country FE
3. Two-way FE (country + year)
4. Two-way FE with interaction terms (`inflation × post_2022`, `financial_account_baseline × post_2022`)

**SE treatment:** Country-clustered. `linearmodels.PanelOLS` handles this natively.

**Note:** `financial_account_baseline` is time-invariant → absorbed by country FE in models 2–4.
Re-introduced via interaction terms in model 4.

In [ ]:
# ── H2: Load data ─────────────────────────────────────────────────────
h2 = pd.read_csv(os.path.join(DATA_DIR, 'h2_diffusion_dataset.csv'))
print(f'H2 loaded: {h2.shape[0]} rows, countries: {h2["country_iso3"].nunique()}, years: {sorted(h2["year"].unique())}')
h2.head()

In [ ]:
# ── H2: Panel data setup ──────────────────────────────────────────────
# TODO: Phase 3.4 — Set MultiIndex (country, year) for linearmodels.
# Construct log transforms, interaction terms.

In [ ]:
# ── H2: Model estimation (4 specifications) ──────────────────────────
# TODO: Phase 3.4 — Pooled OLS → Country FE → Two-way FE → Interactions.
# Country-clustered SE throughout.

In [ ]:
# ── H2: Regression table ──────────────────────────────────────────────
# TODO: Phase 3.4 — Four-column side-by-side table. Save CSV + LaTeX.

In [ ]:
# ── H2: Figure ────────────────────────────────────────────────────────
# TODO: Phase 3.4 — Scatter or choropleth. Save h2_adoption_map.png

---
# H4 — Cost Friction vs. Legacy Rails

**Hypothesis:** On-chain fees are orders of magnitude below legacy remittance costs
at realistic transfer sizes ($200 and $10,000).

**Test:** Parameterized cost comparison across three rails (USDC/ETH, USDT/TRX, SWIFT legacy).
Paired difference-in-means with robust SE.

**Key formula:** `legacy_cost(x) = x × legacy_pct_fee + legacy_flat_fee` vs flat on-chain fee.

In [ ]:
# ── H4: Load data ─────────────────────────────────────────────────────
h4 = pd.read_csv(os.path.join(DATA_DIR, 'h4_infrastructure_cost.csv'))
print(f'H4 loaded: {h4.shape[0]} rows, date range: {h4["month"].iloc[0]} to {h4["month"].iloc[-1]}')
h4.head()

In [ ]:
# ── H4: Cost scenarios ($200 and $10,000) ─────────────────────────────
# TODO: Phase 3.5 — Compute total cost for three rails × two transfer sizes.

In [ ]:
# ── H4: Bar charts ────────────────────────────────────────────────────
# TODO: Phase 3.5 — Grouped bar panels for $200 and $10,000.
# Save: h4_cost_comparison_200.png, h4_cost_comparison_10000.png

In [ ]:
# ── H4: Formal test ───────────────────────────────────────────────────
# TODO: Phase 3.5 — Difference-in-means with robust SE.

In [ ]:
# ── H4: Cost ratio time series ────────────────────────────────────────
# TODO: Phase 3.5 — Legacy/crypto cost ratio over time.
# Save: h4_cost_ratio_timeseries.png

---
# Phase 3 Exit Checklist

Per `Master_Recovery_Roadmap.md` §3.6:

- [ ] All four hypotheses have a formal test with a reported p-value
- [ ] Every headline figure saved to `outputs/figures/` as PNG at 300+ DPI
- [ ] Every regression table saved to `outputs/tables/` in CSV and LaTeX
- [ ] This notebook runs top-to-bottom cleanly on a fresh kernel